## Monter Google Drive (Colab)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Vérifier TensorFlow

In [ ]:
import tensorflow as tf

print("="*50)
print("Version TensorFlow :", tf.__version__)
print("="*50)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU détecté :", gpus)
else:
    print("Aucun GPU détecté")

## import des biblio

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

## definition des chemin dans drive

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Detection_Deforestation_IA"

DATA_PATH = os.path.join(
    BASE_PATH,
    "data"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)


TRAIN_PATH = os.path.join(
    DATA_PATH,
    "train"
)

VALIDATION_PATH = os.path.join(
    DATA_PATH,
    "validation"
)

TEST_PATH = os.path.join(
    DATA_PATH,
    "test"
)


print("Train :", TRAIN_PATH)
print("Validation :", VALIDATION_PATH)
print("Test :", TEST_PATH)
print("Models :", MODEL_PATH) 

## Vérification des dossiers

In [ ]:
print(os.listdir(DATA_PATH))

print("\nTrain :")
print(os.listdir(TRAIN_PATH))

print("\nValidation :")
print(os.listdir(VALIDATION_PATH))

print("\nTest :")
print(os.listdir(TEST_PATH))

## Charger le modèle CNN construit dans Notebook 03

In [ ]:
model = load_model(
    os.path.join(
        MODEL_PATH,
        "cnn_initial.keras"
    )
)

model.summary()

## Création des générateurs d'images

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255
)


validation_datagen = ImageDataGenerator(
    rescale=1./255
)


test_datagen = ImageDataGenerator(
    rescale=1./255
)

## Chargement des images

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary"
)


validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary"
)


test_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

## Vérifier les classes

In [ ]:
print(train_generator.class_indices)

## Création des callbacks

In [ ]:
checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        MODEL_PATH,
        "best_model.keras"
    ),
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)


reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

## Entraînement du CNN

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=30,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)

## Courbe Accuracy

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["accuracy"],
    label="Train"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation"
)

plt.title("Accuracy du modèle")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()
plt.grid()

plt.show()

## Courbe Loss

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Train"
)

plt.plot(
    history.history["val_loss"],
    label="Validation"
)

plt.title("Loss du modèle")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()

plt.show()

## Évaluation sur test

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_generator
)

print("Test Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

## Rapport classification

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


test_generator.reset()


predictions = model.predict(
    test_generator
)


y_pred = (
    predictions > 0.5
).astype(int)


y_true = test_generator.classes


print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "forest",
            "non_forest"
        ]
    )
)

## Matrice de confusion

In [ ]:
import seaborn as sns


cm = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(figsize=(6,5))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=[
        "forest",
        "non_forest"
    ],
    yticklabels=[
        "forest",
        "non_forest"
    ]
)


plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion")


plt.show()

## sauvegarde final du model

In [ ]:
model.save(
    os.path.join(
        MODEL_PATH,
        "deforestation_cnn_final.keras"
    )
)


print(" Modèle final sauvegardé dans Google Drive")